# 03j — SHAP analysis sobre CatBoost tuneado (churn_30d)

Calcula SHAP values sobre 1000 jugadores del test set y genera:
- Beeswarm summary
- Bar global (mean |SHAP|)
- Dependence plots para top 5 features
- Decision plots para 10 ejemplos (5 churners + 5 no-churners)

In [1]:
# [SETUP]
import pandas as pd
import numpy as np
import shap
from catboost import CatBoostClassifier, Pool
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

RANDOM_STATE = 42
TARGET = 'churn_30d'  # Target principal del TFG
N_SAMPLES_SHAP = 1000
TOP_N = 10

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase3_modeling' else Path.cwd()
DATA_QC = PROJECT_ROOT / 'data' / 'data_qc'
DATA_MODELS = PROJECT_ROOT / 'data' / 'data_models'
REPORT_DIR = PROJECT_ROOT / 'informes' / 'fase3_modeling' / '03j_shap'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_PATH = DATA_QC / 'master_table_cutoff_v3_aggressive.parquet'
SPLITS_PATH = DATA_MODELS / 'splits_indices_cutoff.parquet'
MODEL_PATH = DATA_MODELS / f'catboost_tuned_{TARGET}_v3.cbm'

print(f'Target: {TARGET}')
print(f'Modelo: {MODEL_PATH.name}')

Target: churn_30d
Modelo: catboost_tuned_churn_30d_v3.cbm


In [2]:
# [EXEC] Cargar datos
master = pd.read_parquet(MASTER_PATH).reset_index(drop=True)
splits_df = pd.read_parquet(SPLITS_PATH).reset_index(drop=True)
if 'user_id' in splits_df.columns:
    splits_df = splits_df.set_index('user_id').reindex(master['user_id'].values).reset_index()

cat_cols = master.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
cat_cols = [c for c in cat_cols if c != 'user_id']
for c in cat_cols:
    master[c] = master[c].astype(str).fillna('missing').replace('nan', 'missing')

X_full = master.drop(columns=['churn_14d', 'churn_30d', 'user_id'])
test_mask = splits_df['split'].values == 'test'
X_test = X_full[test_mask].reset_index(drop=True)
y_test = master[TARGET].astype(int)[test_mask].reset_index(drop=True)
user_id_test = master.loc[test_mask, 'user_id'].reset_index(drop=True)

# Submuestrear para SHAP
rng = np.random.RandomState(RANDOM_STATE)
shap_idx = rng.choice(len(X_test), size=min(N_SAMPLES_SHAP, len(X_test)), replace=False)
X_shap = X_test.iloc[shap_idx].reset_index(drop=True)
y_shap = y_test.iloc[shap_idx].reset_index(drop=True)
user_id_shap = user_id_test.iloc[shap_idx].reset_index(drop=True)
print(f'X_shap shape: {X_shap.shape}')

model = CatBoostClassifier()
model.load_model(str(MODEL_PATH))

X_shap shape: (1000, 77)


CatBoostClassifier(bagging_temperature=0.005258618458, border_count=55, depth=7, eval_metric='AUC', iterations=1218, l2_leaf_reg=1.740859253, learning_rate=0.0164426694, loss_function='Logloss', min_data_in_leaf=45, od_type='Iter', od_wait=50, random_seed=42, random_strength=0.6822584674, use_best_model=True, verbose=0)

In [3]:
# [EXEC] Calcular SHAP values con CatBoost interno
shap_pool = Pool(X_shap, y_shap, cat_features=cat_cols)
shap_values_raw = model.get_feature_importance(shap_pool, type='ShapValues')
# CatBoost devuelve (n_samples, n_features+1): la última columna es expected_value (base)
expected_value = shap_values_raw[:, -1].mean()
shap_values = shap_values_raw[:, :-1]
feature_names = X_shap.columns.tolist()
print(f'SHAP values shape: {shap_values.shape} | expected_value: {expected_value:.4f}')

# Convertir SHAP values a sklearn-compatible Explanation
explanation = shap.Explanation(
    values=shap_values,
    base_values=np.full(shap_values.shape[0], expected_value),
    data=X_shap.values,
    feature_names=feature_names,
)

SHAP values shape: (1000, 77) | expected_value: 0.0363


In [4]:
# [ANALYSIS] Mean |SHAP| → top 10 features
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_features = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_shap': mean_abs_shap,
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
top_features.to_csv(REPORT_DIR / 'top_features_shap.csv', index=False)
print('Top 10 features por |SHAP| medio:')
print(top_features.head(10).to_string(index=False))
TOP_5_NAMES = top_features['feature'].head(5).tolist()

Top 10 features por |SHAP| medio:
                      feature  mean_abs_shap
device_days_since_last_active       0.363230
   reward_last_claim_days_ago       0.274747
   char_last_updated_days_ago       0.214313
   items_days_since_last_item       0.198704
       reward_current_day_max       0.185319
    coll_days_since_last_item       0.129668
                      country       0.113945
     user_created_at_days_ago       0.085469
    reward_sets_completed_max       0.080782
            reward_claim_rate       0.079370


In [5]:
# [ANALYSIS] Beeswarm + bar global
# Para beeswarm necesitamos features numéricas; convertir categóricas a códigos para que SHAP las plotee
X_shap_numeric = X_shap.copy()
for c in cat_cols:
    X_shap_numeric[c] = pd.Categorical(X_shap_numeric[c]).codes
explanation_numeric = shap.Explanation(
    values=shap_values,
    base_values=np.full(shap_values.shape[0], expected_value),
    data=X_shap_numeric.values,
    feature_names=feature_names,
)

fig = plt.figure(figsize=(10, 8))
shap.plots.beeswarm(explanation_numeric, max_display=15, show=False)
plt.title(f'SHAP beeswarm — {TARGET}')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'shap_summary_beeswarm.png', dpi=120, bbox_inches='tight')
plt.close()

fig = plt.figure(figsize=(8, 7))
shap.plots.bar(explanation_numeric, max_display=15, show=False)
plt.title(f'SHAP bar global — {TARGET}')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'shap_bar_global.png', dpi=120, bbox_inches='tight')
plt.close()
print('beeswarm + bar guardados')

beeswarm + bar guardados


In [6]:
# [ANALYSIS] Dependence plots top 5
for feat in TOP_5_NAMES:
    fig, ax = plt.subplots(figsize=(8, 5))
    feat_idx = feature_names.index(feat)
    feat_values = X_shap_numeric.iloc[:, feat_idx].values
    shap_vals = shap_values[:, feat_idx]
    ax.scatter(feat_values, shap_vals, c=shap_vals, cmap='coolwarm', s=15, alpha=0.6)
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
    ax.set_xlabel(feat)
    ax.set_ylabel('SHAP value')
    ax.set_title(f'Dependence: {feat} → {TARGET}')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    safe_name = feat.replace('/', '_')
    plt.savefig(REPORT_DIR / f'shap_dependence_{safe_name}.png', dpi=120, bbox_inches='tight')
    plt.close()
print(f'{len(TOP_5_NAMES)} dependence plots guardados')

5 dependence plots guardados


In [7]:
# [ANALYSIS] Decision plots para 10 ejemplos (5 churners + 5 no-churners predichos correctamente)
y_pred_proba_shap = model.predict_proba(shap_pool)[:, 1]
y_pred_shap = (y_pred_proba_shap >= 0.5).astype(int)
correct = (y_pred_shap == y_shap.values)
churners_correct_idx = np.where((y_shap.values == 1) & correct)[0][:5]
active_correct_idx = np.where((y_shap.values == 0) & correct)[0][:5]
examples_idx = np.concatenate([churners_correct_idx, active_correct_idx])
labels = ['churner']*len(churners_correct_idx) + ['active']*len(active_correct_idx)

fig, ax = plt.subplots(figsize=(10, 7))
shap.decision_plot(
    expected_value,
    shap_values[examples_idx],
    X_shap_numeric.iloc[examples_idx],
    feature_names=feature_names,
    show=False,
    legend_labels=labels,
    ignore_warnings=True,
)
plt.title(f'Decision plot — 10 ejemplos correctos ({TARGET})')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'shap_decision_examples.png', dpi=120, bbox_inches='tight')
plt.close()

# Guardar tabla de ejemplos
examples_df = pd.DataFrame({
    'user_id': user_id_shap.iloc[examples_idx].values,
    'y_true': y_shap.iloc[examples_idx].values,
    'y_pred': y_pred_shap[examples_idx],
    'y_pred_proba': y_pred_proba_shap[examples_idx],
    'kind': labels,
})
examples_df.to_csv(REPORT_DIR / 'shap_decision_examples.csv', index=False)
print(f'decision_plot con {len(examples_idx)} ejemplos guardado')

decision_plot con 10 ejemplos guardado


In [8]:
# [REPORT] shap_summary.md
lines = [
    f'# SHAP analysis — CatBoost {TARGET}',
    '',
    f'**Fecha:** {datetime.now().strftime("%Y-%m-%d %H:%M")}',
    f'**Modelo:** catboost_tuned_{TARGET}_v3.cbm',
    f'**Muestra:** {N_SAMPLES_SHAP} jugadores aleatorios del test set ({len(X_test):,} totales)',
    '',
    f'## Top {TOP_N} features por |SHAP| medio',
    '',
    '| Rank | Feature | mean(|SHAP|) |',
    '|---:|---|---:|',
]
for i, row in top_features.head(TOP_N).iterrows():
    lines.append(f'| {i+1} | `{row["feature"]}` | {row["mean_abs_shap"]:.4f} |')

lines += [
    '',
    '## Outputs',
    '',
    '- `top_features_shap.csv` — ranking completo por |SHAP| medio',
    '- `shap_summary_beeswarm.png` — distribución de SHAP por feature',
    '- `shap_bar_global.png` — magnitud media de SHAP por feature',
    '- `shap_dependence_<feat>.png` × 5 — efecto marginal de las top 5 features',
    '- `shap_decision_examples.png` — 10 ejemplos (5 churners + 5 activos predichos OK)',
    '- `shap_decision_examples.csv` — IDs y predicciones de los ejemplos',
    '',
    '## Interpretación',
    '',
    'Las features más informativas para predecir churn 30d son:',
    '',
]
for i, row in top_features.head(5).iterrows():
    lines.append(f'{i+1}. **`{row["feature"]}`** (|SHAP|={row["mean_abs_shap"]:.4f}) — ver `shap_dependence_{row["feature"]}.png` para el efecto marginal.')

lines += [
    '',
    'Implicaciones para diseño de contramedidas: priorizar intervenciones que muevan',
    'las top 3-5 features hacia el rango asociado con menor SHAP (menor probabilidad',
    'de churn). Ver dependence plots para identificar valores umbral.',
]

with open(REPORT_DIR / 'shap_summary.md', 'w') as f:
    f.write('\n'.join(lines))
print('shap_summary.md guardado')

shap_summary.md guardado
